In [ ]:
import pandas as pd

# df = pd.read_csv(r'D:\KAIM-9\Week-1\news-sentiment-analysis\data\raw_analyst_ratings.csv')

# Assign the loaded DataFrame 'df' to 'df_news' for consistent analysis
df_news = pd.read_csv(r'D:\KAIM-9\Week-1\news-sentiment-analysis\data\raw_analyst_ratings.csv')

if 'df_news' in locals() and not df_news.empty:
    print("\n--- Performing Descriptive Statistics ---")

    # a) Obtain basic statistics for textual lengths (e.g., headline character count distribution)
    if 'headline' in df_news.columns:
        df_news['headline_length'] = df_news['headline'].apply(len)
        print("\n1. Headline Character Count Distribution:")
        print(df_news['headline_length'].describe())
    else:
        print("\nWarning: 'headline' column not found for textual length analysis.")

    # b) Count articles per publisher to identify which sources are most active
    if 'publisher' in df_news.columns:
        print("\n2. Articles per Publisher:")
        print(df_news['publisher'].value_counts())
    else:
        print("\nWarning: 'publisher' column not found for articles per publisher analysis.")

    # c) Analyze publication dates to identify trends over time
    if 'date' in df_news.columns:
        print("\n3. Publication Dates Trends (Articles per Day):")
        try:
            # Use errors='coerce' to turn unparseable dates into NaT and ensure datetime dtype
            # Also add utc=True as suggested by the FutureWarning for mixed time zones
            df_news['date'] = pd.to_datetime(df_news['date'], format='mixed', dayfirst=False, errors='coerce', utc=True)
            # Drop NaT values before counting dates to avoid errors if any unparseable dates exist
            print(df_news['date'].dropna().dt.date.value_counts().sort_index())
        except Exception as e:
            print(f"Could not convert 'date' to datetime. Error: {e}")
    else:
        print("\nWarning: 'date' column not found for date trend analysis.")
else:
    print("\nError: df_news is empty or not properly loaded. Please ensure your FNSPID data is in 'df_news'.")


--- Performing Descriptive Statistics ---

1. Headline Character Count Distribution:
count    1.407328e+06
mean     7.312051e+01
std      4.073531e+01
min      3.000000e+00
25%      4.700000e+01
50%      6.400000e+01
75%      8.700000e+01
max      5.120000e+02
Name: headline_length, dtype: float64

2. Articles per Publisher:
publisher
Paul Quintaro        228373
Lisa Levin           186979
Benzinga Newsdesk    150484
Charles Gross         96732
Monica Gerson         82380
                      ...  
MoneyGeek                 1
muathe                    1
Robert Morris             1
LeftCoastHedgie           1
Jeremie Capron            1
Name: count, Length: 1034, dtype: int64

3. Publication Dates Trends (Articles per Day):
date
2009-02-14      1
2009-04-27      2
2009-04-29      1
2009-05-22      1
2009-05-27      6
             ... 
2020-06-07     25
2020-06-08    765
2020-06-09    803
2020-06-10    807
2020-06-11    544
Name: count, Length: 3955, dtype: int64


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

if 'headline' in df_news.columns:
    print("\n--- TF-IDF Analysis (Top 20 Keywords) ---")
    
    # For faster processing, especially with large datasets, work on a sample
    # You can adjust the sample_size as needed.
    sample_size = 100000 # Using 100,000 headlines for quicker analysis
    if len(df_news) > sample_size:
        sampled_headlines = df_news['headline'].sample(n=sample_size, random_state=42).tolist()
        print(f"Analyzing a sample of {sample_size} headlines.")
    else:
        sampled_headlines = df_news['headline'].tolist()
        print(f"Analyzing all {len(sampled_headlines)} headlines.")

    # Initialize TfidfVectorizer
    tfidf_vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
    tfidf_matrix = tfidf_vectorizer.fit_transform(sampled_headlines)

    # Get feature names (words)
    tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

    # Sum TF-IDF scores for each word across all headlines
    tfidf_sums = tfidf_matrix.sum(axis=0)

    # Convert to 1D numpy array ensuring correct shape
    tfidf_sums_array = np.asarray(tfidf_sums).flatten()

    # Get indices of words with highest TF-IDF scores
    most_important_words_indices = tfidf_sums_array.argsort()[::-1]

    # Print top 20 keywords by TF-IDF score
    if tfidf_sums_array.size > 0:
        for i in most_important_words_indices[:20]:
            print(f"{tfidf_feature_names[i]}: {tfidf_sums_array[i]:.2f}")
    else:
        print("No important keywords found with TF-IDF (tfidf_sums is empty).")
else:
    print("Warning: 'headline' column not found for TF-IDF analysis.")